In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix

In [2]:
DATA_DIR = Path("../data/processed")

ratings = pd.read_csv(DATA_DIR / "ratings_clean.csv")
movies = pd.read_csv(DATA_DIR / "movies_clean.csv")

genre_features = pd.read_csv(
    DATA_DIR / "features/movie_genre_features.csv"
)

print("Ratings:", ratings.shape)
print("Movies:", movies.shape)
print("Genre features:", genre_features.shape)

Ratings: (1000209, 5)
Movies: (3883, 3)
Genre features: (3883, 19)


In [3]:
train_parts = []
test_parts = []

for user_id, group in ratings.groupby("UserID"):
    group = group.sample(frac=1, random_state=42)

    test_size = max(1, int(len(group) * 0.2))

    test_parts.append(group.iloc[:test_size])
    train_parts.append(group.iloc[test_size:])

train_ratings = pd.concat(train_parts).reset_index(drop=True)
test_ratings = pd.concat(test_parts).reset_index(drop=True)

print("Training ratings:", train_ratings.shape)
print("Testing ratings:", test_ratings.shape)

Training ratings: (802553, 5)
Testing ratings: (197656, 5)


In [4]:
print("Training users:", train_ratings["UserID"].nunique())
print("Testing users:", test_ratings["UserID"].nunique())

print(
    "Training percentage:",
    round(len(train_ratings) / len(ratings) * 100, 2),
    "%"
)

print(
    "Testing percentage:",
    round(len(test_ratings) / len(ratings) * 100, 2),
    "%"
)

Training users: 6040
Testing users: 6040
Training percentage: 80.24 %
Testing percentage: 19.76 %


In [5]:
print("Genre feature columns:")
print(genre_features.columns.tolist())

display(genre_features.head())

Genre feature columns:
['MovieID', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


,MovieID,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
2,3,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
3,4,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
4,5,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [6]:
genre_matrix = genre_features.copy()

if "MovieID" not in genre_matrix.columns:
    raise ValueError("MovieID column is missing from genre features.")

genre_matrix = genre_matrix.set_index("MovieID")

genre_matrix = genre_matrix.apply(pd.to_numeric)

genre_matrix.head()

,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
MovieID,,,,,,,,,,,,,,,,,,
1,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [7]:
train_positive = train_ratings[
    train_ratings["Rating"] >= 4
]

test_positive = test_ratings[
    test_ratings["Rating"] >= 4
]

print("Positive training ratings:", len(train_positive))
print("Positive testing ratings:", len(test_positive))

Positive training ratings: 461478
Positive testing ratings: 113803


In [8]:
user_profiles = {}

for user_id, group in train_positive.groupby("UserID"):

    liked_movie_ids = group["MovieID"].tolist()

    available_movies = genre_matrix.index.intersection(
        liked_movie_ids
    )

    if len(available_movies) == 0:
        continue

    profile = genre_matrix.loc[available_movies].mean(axis=0)

    user_profiles[user_id] = profile

print("User profiles created:", len(user_profiles))

User profiles created: 6038


In [9]:
test_relevant = (
    test_positive.groupby("UserID")["MovieID"]
    .apply(set)
    .to_dict()
)

train_watched = (
    train_ratings.groupby("UserID")["MovieID"]
    .apply(set)
    .to_dict()
)


def evaluate_content_based(k=10):

    precisions = []
    recalls = []

    for user_id, relevant_movies in test_relevant.items():

        if user_id not in user_profiles:
            continue

        profile = user_profiles[user_id].to_numpy().reshape(1, -1)

        watched_movies = train_watched.get(user_id, set())

        candidate_features = genre_matrix.drop(
            index=list(watched_movies),
            errors="ignore"
        )

        if candidate_features.empty:
            continue

        scores = cosine_similarity(
            profile,
            candidate_features
        )[0]

        top_indices = np.argsort(scores)[::-1][:k]

        recommended_movies = set(
            candidate_features.index[top_indices]
        )

        relevant_recommended = (
            recommended_movies.intersection(relevant_movies)
        )

        precision = len(relevant_recommended) / k

        recall = (
            len(relevant_recommended) / len(relevant_movies)
        )

        precisions.append(precision)
        recalls.append(recall)

    return {
        "Precision@10": np.mean(precisions),
        "Recall@10": np.mean(recalls),
        "Evaluated Users": len(precisions)
    }

In [10]:
content_results = evaluate_content_based(k=10)

content_results

{'Precision@10': np.float64(0.025440638510143),
 'Recall@10': np.float64(0.018781261996204925),
 'Evaluated Users': 6014}

In [11]:
user_movie_train = train_ratings.pivot(
    index="UserID",
    columns="MovieID",
    values="Rating"
)

print("Training matrix shape:", user_movie_train.shape)

Training matrix shape: (6040, 3680)


In [12]:
train_user_means = train_ratings.groupby("UserID")["Rating"].mean()

train_global_mean = train_ratings["Rating"].mean()

print("Global average rating:", round(train_global_mean, 3))
print("Number of users:", len(train_user_means))

Global average rating: 3.581
Number of users: 6040


In [13]:
user_ids_train = user_movie_train.index.tolist()
movie_ids_train = user_movie_train.columns.tolist()

user_index_map = {
    user_id: index
    for index, user_id in enumerate(user_ids_train)
}

movie_index_map = {
    movie_id: index
    for index, movie_id in enumerate(movie_ids_train)
}

centered_train = train_ratings.copy()

centered_train["UserMean"] = centered_train["UserID"].map(
    train_user_means
)

centered_train["CenteredRating"] = (
    centered_train["Rating"] - centered_train["UserMean"]
)

row_indices = centered_train["UserID"].map(
    user_index_map
).to_numpy()

column_indices = centered_train["MovieID"].map(
    movie_index_map
).to_numpy()

values = centered_train["CenteredRating"].to_numpy(
    dtype=np.float32
)

centered_train_matrix = csr_matrix(
    (values, (row_indices, column_indices)),
    shape=(len(user_ids_train), len(movie_ids_train))
)

print("Centered matrix shape:", centered_train_matrix.shape)
print("Stored ratings:", centered_train_matrix.nnz)

Centered matrix shape: (6040, 3680)
Stored ratings: 802553


In [14]:
train_user_similarity = cosine_similarity(
    centered_train_matrix,
    dense_output=False
)

print("Similarity matrix shape:", train_user_similarity.shape)

Similarity matrix shape: (6040, 6040)


In [15]:
def predict_rating(user_id, movie_id, neighbor_count=20):

    if user_id not in user_index_map:
        return train_global_mean

    if movie_id not in movie_index_map:
        return train_user_means.get(user_id, train_global_mean)

    user_index = user_index_map[user_id]
    movie_index = movie_index_map[movie_id]

    similarities = train_user_similarity.getrow(user_index)

    neighbor_indices = similarities.indices
    neighbor_scores = similarities.data

    rated_users = train_ratings[
        train_ratings["MovieID"] == movie_id
    ]["UserID"]

    rated_user_indices = {
        user_index_map[uid]
        for uid in rated_users
        if uid in user_index_map
    }

    valid = (
        (neighbor_indices != user_index)
        & (neighbor_scores > 0)
        & np.isin(neighbor_indices, list(rated_user_indices))
    )

    neighbor_indices = neighbor_indices[valid]
    neighbor_scores = neighbor_scores[valid]

    if len(neighbor_indices) == 0:
        return train_user_means.get(user_id, train_global_mean)

    top_positions = np.argsort(neighbor_scores)[::-1][:neighbor_count]

    selected_indices = neighbor_indices[top_positions]
    selected_scores = neighbor_scores[top_positions]

    selected_user_ids = [
        user_ids_train[i] for i in selected_indices
    ]

    neighbor_ratings = train_ratings[
        (train_ratings["UserID"].isin(selected_user_ids))
        & (train_ratings["MovieID"] == movie_id)
    ].set_index("UserID")

    weighted_deviations = []

    for neighbor_id, similarity in zip(
        selected_user_ids, selected_scores
    ):
        rating = neighbor_ratings.loc[neighbor_id, "Rating"]
        neighbor_mean = train_user_means[neighbor_id]

        weighted_deviations.append(
            similarity * (rating - neighbor_mean)
        )

    prediction = (
        train_user_means[user_id]
        + sum(weighted_deviations) / np.sum(selected_scores)
    )

    return float(np.clip(prediction, 1, 5))

In [16]:
actual_ratings = []
predicted_ratings = []

for row in test_ratings.itertuples(index=False):

    prediction = predict_rating(
        user_id=row.UserID,
        movie_id=row.MovieID
    )

    actual_ratings.append(row.Rating)
    predicted_ratings.append(prediction)

rmse = np.sqrt(
    mean_squared_error(actual_ratings, predicted_ratings)
)

mae = mean_absolute_error(
    actual_ratings, predicted_ratings
)

print("RMSE:", round(rmse, 4))
print("MAE:", round(mae, 4))

RMSE: 0.8945
MAE: 0.696


In [28]:
rating_presence = centered_train_matrix.copy()
rating_presence.data = np.ones_like(rating_presence.data)

print("Rating presence matrix:", rating_presence.shape)

Rating presence matrix: (6040, 3680)


In [29]:
from scipy.sparse import lil_matrix

neighbor_count = 20

top20_similarity = lil_matrix(
    train_user_similarity.shape,
    dtype=np.float32
)

for user_index in range(train_user_similarity.shape[0]):

    row = train_user_similarity.getrow(user_index)

    indices = row.indices
    scores = row.data

    valid = (indices != user_index) & (scores > 0)

    indices = indices[valid]
    scores = scores[valid]

    top_positions = np.argsort(scores)[::-1][:neighbor_count]

    top20_similarity[user_index, indices[top_positions]] = (
        scores[top_positions]
    )

top20_similarity = top20_similarity.tocsr()

print("Top-20 similarity matrix:", top20_similarity.shape)
print("Stored similarities:", top20_similarity.nnz)

Top-20 similarity matrix: (6040, 6040)
Stored similarities: 120800


In [30]:
weighted_deviations_matrix = top20_similarity.dot(
    centered_train_matrix
).tocsr()

similarity_weights_matrix = top20_similarity.dot(
    rating_presence
).tocsr()

print("Prediction matrices prepared.")

Prediction matrices prepared.


In [31]:
def get_fast_predictions(user_id):

    if user_id not in user_index_map:
        return None

    user_index = user_index_map[user_id]

    numerator = weighted_deviations_matrix.getrow(
        user_index
    ).toarray()[0]

    denominator = similarity_weights_matrix.getrow(
        user_index
    ).toarray()[0]

    predictions = np.full(
        len(movie_ids_train),
        train_user_means[user_id],
        dtype=np.float32
    )

    valid = denominator > 0

    predictions[valid] = (
        train_user_means[user_id]
        + numerator[valid] / denominator[valid]
    )

    return np.clip(predictions, 1, 5)

In [32]:
movie_array = np.array(movie_ids_train)

print("Total candidate movies:", len(movie_array))

Total candidate movies: 3680


In [33]:
def evaluate_collaborative_filtering(k=10):

    precisions = []
    recalls = []

    for user_id, relevant_movies in test_relevant.items():

        if user_id not in user_index_map:
            continue

        predictions = get_fast_predictions(user_id)

        watched_movies = train_watched.get(user_id, set())

        candidate_mask = np.array([
            movie_id not in watched_movies
            for movie_id in movie_array
        ])

        candidate_indices = np.where(candidate_mask)[0]

        if len(candidate_indices) < k:
            continue

        candidate_scores = predictions[candidate_indices]

        top_positions = np.argsort(candidate_scores)[::-1][:k]

        top_movie_indices = candidate_indices[top_positions]

        recommended_movies = set(
            movie_array[top_movie_indices]
        )

        relevant_recommended = recommended_movies.intersection(
            relevant_movies
        )

        precisions.append(len(relevant_recommended) / k)

        recalls.append(
            len(relevant_recommended) / len(relevant_movies)
        )

    return {
        "Precision@10": np.mean(precisions),
        "Recall@10": np.mean(recalls),
        "Evaluated Users": len(precisions)
    }

In [34]:
cf_ranking_results = evaluate_collaborative_filtering(k=10)

cf_ranking_results

{'Precision@10': np.float64(0.018822746923844363),
 'Recall@10': np.float64(0.011821831459915586),
 'Evaluated Users': 6014}

In [35]:
comparison = pd.DataFrame([
    {
        "Model": "Content-Based Filtering",
        "Precision@10": content_results["Precision@10"],
        "Recall@10": content_results["Recall@10"],
        "RMSE": np.nan,
        "MAE": np.nan
    },
    {
        "Model": "Collaborative Filtering",
        "Precision@10": cf_ranking_results["Precision@10"],
        "Recall@10": cf_ranking_results["Recall@10"],
        "RMSE": rmse,
        "MAE": mae
    }
])

comparison

,Model,Precision@10,Recall@10,RMSE,MAE
0,Content-Based Filtering,0.025441,0.018781,NaN,NaN
1,Collaborative Filtering,0.018823,0.011822,0.894543,0.696043


In [36]:
REPORT_DIR = Path("../reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

comparison.to_csv(
    REPORT_DIR / "model_evaluation_results.csv",
    index=False
)

print("Evaluation results saved successfully.")

Evaluation results saved successfully.
